In [1]:
import pandas as pd
import numpy as np

# NORC

In [2]:
norc_raw = pd.read_csv("../data/raw-data/NORC/31122410.csv")

display(norc_raw)

,SU_ID,POLLCLOSE_STATE_WEIGHT,FINALVOTE_STATE_WEIGHT,POLLCLOSE_NATIONAL_WEIGHT,FINALVOTE_NATIONAL_WEIGHT,MODE,P_STATE,STATENUM,LVB,LIKELYVOTER,...,BORNAGAIN,BORNAGAINSTATE,ATTENDANCE,PARENTUS,CATOWNER,DOGOWNER,TIKTOKUSER,LGBT,HOUSING,SIZEPLACE
0,1000001,546.798136,0.357366,442.024098,0.250968,(2) Web,(HI) Hawaii,(11) Hawaii,(1) Definitely will vote,(1) Likely voter,...,(2) No,(2) No,,(2) No,(2) No,(1) Yes,(1) Yes,(2) No,(2) Rented for cash,(2) Suburban
1,1000002,139.874024,0.231562,105.968208,0.076913,(2) Web,(ME) Maine,(21) Maine,(1) Definitely will vote,(1) Likely voter,...,(2) No,(2) No,,(2) No,(2) No,(2) No,(2) No,(2) No,(2) Rented for cash,(3) Small town
2,1000003,1434.081847,1.508009,1274.137585,0.950696,(2) Web,(MN) Minnesota,(23) Minnesota,(1) Definitely will vote,(1) Likely voter,...,,,,,,,,,(1) Owned or being bought by you or someone in...,(4) Rural
3,1000004,634.071275,0.380504,512.574504,0.267217,(2) Web,(HI) Hawaii,(11) Hawaii,(5) Already voted,(1) Likely voter,...,,,,,,,,,(1) Owned or being bought by you or someone in...,(2) Suburban
4,1000005,1.241451,0.001997,0.822831,0.000701,(2) Web,(NV) Nevada,(33) Nevada,(1) Definitely will vote,(1) Likely voter,...,,,,,,,,,(2) Rented for cash,(3) Small town
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139933,1139934,494.636477,0.317270,365.905264,0.248358,(2) Web,(KY) Kentucky,(17) Kentucky,(1) Definitely will vote,(1) Likely voter,...,,,,,,,,,(1) Owned or being bought by you or someone in...,(4) Rural
139934,1139935,902.328170,1.982933,725.383326,0.681543,(2) Web,(MT) Montana,(26) Montana,(1) Definitely will vote,(1) Likely voter,...,(2) No,(2) No,,(2) No,(1) Yes,(1) Yes,(1) Yes,(1) Yes,(2) Rented for cash,(2) Suburban
139935,1139936,511.050649,0.450388,406.456748,0.282288,(2) Web,(AZ) Arizona,(4) Arizona,(1) Definitely will vote,(1) Likely voter,...,(1) Yes,(1) Yes,,(2) No,(2) No,(2) No,(1) Yes,(2) No,(1) Owned or being bought by you or someone in...,(1) Urban
139936,1139937,4832.216760,1.084959,3804.728285,2.281002,(2) Web,(AL) Alabama,(2) Alabama,(1) Definitely will vote,(1) Likely voter,...,,,(2) A few times a year or less,,,,,,(1) Owned or being bought by you or someone in...,(2) Suburban


### Possible Visualization 1

* Comparing how men vs. women answered various opinion questions of Trump vs. Harris
* Comparing how men vs. women view different politicians (who have dynamic views and personal circumstances that may influence opinion) - is there a difference where even if women tend to be more liberal do they still support conservative politicians?
  * Is there a discrepancy between how men/women view issues and how they approve of different politicians?

In [3]:
display(norc_raw.dtypes)

SU_ID                          int64
POLLCLOSE_STATE_WEIGHT       float64
FINALVOTE_STATE_WEIGHT       float64
POLLCLOSE_NATIONAL_WEIGHT    float64
FINALVOTE_NATIONAL_WEIGHT    float64
                              ...   
DOGOWNER                      object
TIKTOKUSER                    object
LGBT                          object
HOUSING                       object
SIZEPLACE                     object
Length: 466, dtype: object

# Import SAVE Act Data

Use existing SAVE Act data from CAP, but re-create their estimates of women whose names are different on their birth certificates by using ACS data like they did.

In [4]:
raw_acs_df = pd.read_csv("../data/raw-data/CAP/ACS_sex_by_age.csv")
relevant_cols = [col for col in raw_acs_df.columns if 'Estimate' in col or 'Label' in col]
acs_df = raw_acs_df[relevant_cols]

In [5]:
display(acs_df)

,Label (Grouping),Alabama!!Estimate,Alaska!!Estimate,Arizona!!Estimate,Arkansas!!Estimate,California!!Estimate,Colorado!!Estimate,Connecticut!!Estimate,Delaware!!Estimate,District of Columbia!!Estimate,...,Tennessee!!Estimate,Texas!!Estimate,Utah!!Estimate,Vermont!!Estimate,Virginia!!Estimate,Washington!!Estimate,West Virginia!!Estimate,Wisconsin!!Estimate,Wyoming!!Estimate,Puerto Rico!!Estimate
0,Total:,"5,157,699","740,133","7,582,384","3,088,354","39,431,263","5,957,494","3,675,069","1,051,917","702,250",...,"7,227,750","31,290,831","3,503,613","648,493","8,811,195","7,958,180","1,769,979","5,960,975","587,618","3,203,295"
1,Male:,"2,490,551","390,301","3,782,883","1,517,526","19,675,103","3,016,719","1,804,896","506,763","332,062",...,"3,544,910","15,608,512","1,778,121","320,167","4,356,641","4,009,969","881,512","2,984,118","301,319","1,516,882"
2,Under 5 years,"141,881","23,437","200,775","89,764","1,061,708","156,618","93,454","26,707","19,495",...,"207,418","991,727","115,953","13,467","245,420","217,786","42,353","154,111","15,266","48,205"
3,5 to 9 years,"161,224","22,605","207,831","97,420","1,159,051","167,233","102,579","28,485","19,365",...,"214,054","1,072,050","126,522","16,647","265,809","237,146","42,945","173,215","15,871","62,986"
4,10 to 14 years,"164,873","27,855","246,928","104,574","1,279,071","180,097","107,177","32,523","17,192",...,"238,684","1,141,156","145,413","15,786","283,971","243,372","57,247","186,403","20,765","83,132"
5,15 to 17 years,"103,995","15,902","153,345","64,562","806,056","117,606","70,697","18,913","9,048",...,"149,058","707,123","93,309","10,411","173,870","153,077","35,519","119,714","14,899","55,370"
6,18 and 19 years,"73,450","10,010","107,786","43,518","544,659","81,757","49,599","12,468","8,739",...,"90,488","460,613","56,325","9,896","130,191","96,960","23,867","79,218","8,326","43,228"
7,20 years,"37,286","5,239","56,003","21,655","269,065","37,515","25,446","6,487","4,295",...,"46,351","215,717","31,034","4,608","59,554","48,669","10,962","40,600","3,862","23,538"
8,21 years,"33,082","5,648","51,119","18,458","277,889","38,676","24,660","6,178","3,293",...,"43,707","221,504","33,032","4,532","59,391","45,141","12,359","37,268","4,503","20,984"
9,22 to 24 years,"96,642","18,642","162,336","63,034","774,037","126,757","74,273","20,636","14,945",...,"156,010","677,225","93,688","12,265","177,899","161,418","32,659","126,666","12,800","59,739"
